In [4]:
# import the functions from collector.py
import sys
import os
sys.path.append(os.path.abspath(".."))

from bjarne_api.collector_new import Fetcher
import pandas as pd

In [5]:
### FILTER FUNCTION FOR ICE/IC TRAINS ONLY ###
def filter_train_type(df):
    output_df = df[df["train_type"].str.contains("ICE|IC", case=False, na=False)].copy()
    return output_df

In [6]:
### FUNCTION TO FIND POSSIBLE DESTINATIONS FROM A GIVEN STATION ###
def possible_destinations(df, station_name):
    possible_destinations = []

    for ride_id, train_group in df.groupby("ride_id"):
        
        # find row where current_station is station_name
        start_row = train_group[train_group["station_current"] == station_name]
        
        # If the train actually stops at our station:
        if not start_row.empty:
            station_time = start_row["departure_real"].iloc[0]
            
            # get stations where actual_arrival is greater than station_time
            later_stops = train_group[train_group["arrival_real"] > station_time]
            
            # get station names and add to possible_destinations
            stations_names = later_stops["station_current"].tolist()
            possible_destinations.extend(stations_names)
            
    # use set() to remove duplicates
    return set(possible_destinations)

In [7]:
import pandas as pd

### FUNCTION TO GET CONNECTION BETWEEN TWO STATIONS ###
def get_connection(df, station_start, station_dest):
    
    # check if station_dest is reachable from station_start
    valid_destinations = possible_destinations(df, station_start)

    if station_dest not in valid_destinations:
        print(f"{station_dest} not reachable from {station_start} with a train departing in the next 60 minutes.")
        print(f"Possible options are: {sorted(list(valid_destinations))}")
        return None

    
    suited_rides = []

    for ride_id, train_group in df.groupby("ride_id"):
        start_row = train_group[train_group["station_current"] == station_start]
        
        if not start_row.empty:
            station_start_time = start_row["departure_real"].iloc[0]
            
            # get stations coming after station_start_time
            later_stops = train_group[train_group["arrival_real"] > station_start_time]
            
            # check: is station_dest in later_stops?
            if station_dest in later_stops["station_current"].values:
                # add ride_id to suited_rides
                suited_rides.append(train_group)

    # combine all suited_rides into one df
    if suited_rides:
        df_trip = pd.concat(suited_rides, ignore_index=True)
        return df_trip
    else:
        print(f"No current connections found.")
        return None

In [8]:
### FUNCTION TO GET DF WITH RIDES BETWEEN TWO STATIONS ###
def get_connections_between_stations(df, start_station, end_station):

    df_temp = filter_train_type(df)
    
    return get_connection(df_temp, start_station, end_station)

In [9]:
### DEFINE JOURNEY
input = "Berlin Hbf" 
destination = "Göttingen"


### GET INFOS ON ALL RIDES
# Ausführung im Hauptprogramm 
if __name__ == "__main__":
    fetcher = Fetcher()
    station_name = input
    
    print(f"Suche Station ID für {station_name}...")
    station_id = fetcher.get_station_id(station_name)
    
    if station_id:
        print(f"Station ID gefunden: {station_id}. Suche Fernverkehrsabfahrten (nächste 60min)...")
        departures = fetcher.stations_details(station_id)
        
        if departures:
            print(f"{len(departures)} Abfahrten gefunden. Lade Details...")
            
            all_trips_dfs = []
            
            
            for i, departure in enumerate(departures, start=1):
                if "tripId" in departure:
                    trip_id = departure["tripId"]
                    line_name = departure.get("line", {}).get("name", "Unbekannt")
                    
                    print(f"Lade Trip {i}/{len(departures)}: {line_name}")
                    
                    trip_data = fetcher.trip_details(trip_id)
                    if trip_data is not None:
                        df_trip = fetcher.create_dataframe(trip_data, ride_id=i)
                    
                    if not df_trip.empty:
                        all_trips_dfs.append(df_trip)
                        
                    else: 
                        print(f"Details für Trip {line_name} konnten nicht verarbeitete werden")
            
            # Zusammenfügen aller DataFrames
            if all_trips_dfs:
                final_df = pd.concat(all_trips_dfs, ignore_index=True)
                print("\n--- FERTIG ---")
                print(final_df)
                print(f"\nGesamtanzahl Haltepunkte analysiert: {len(final_df)}")
                print(f"Anzahl Züge: {final_df['ride_id'].nunique()}")
            else:
                print("Konnte keine Trip-Details verarbeiten.")
        else:
            print("Keine passenden Abfahrten im Zeitraum gefunden.")
    else:
        print("Station nicht gefunden.")



df_trip = get_connections_between_stations(final_df, input, destination)
df_trip = df_trip.drop(columns = ["current_delay"])

Suche Station ID für Berlin Hbf...
Station ID gefunden: 8011160. Suche Fernverkehrsabfahrten (nächste 60min)...
14 Abfahrten gefunden. Lade Details...
Lade Trip 1/14: ICE 1163
Lade Trip 2/14: ICE 140
Lade Trip 3/14: IC 2271
Lade Trip 4/14: ICE 379
Lade Trip 5/14: ICE 603
Lade Trip 6/14: FLX 1328
Lade Trip 7/14: BUS 40249
Lade Trip 8/14: ICE 1103
Lade Trip 9/14: ICE 2824
Lade Trip 10/14: ICE 1904
Lade Trip 11/14: ICE 544
Lade Trip 12/14: ICE 838
Lade Trip 13/14: ICE 935
Lade Trip 14/14: ICE 2933

--- FERTIG ---
     ride_id train_name train_type         station_start     station_dest  \
0          1   ICE 1163        ICE  Berlin Gesundbrunnen    Stuttgart Hbf   
1          1   ICE 1163        ICE  Berlin Gesundbrunnen    Stuttgart Hbf   
2          1   ICE 1163        ICE  Berlin Gesundbrunnen    Stuttgart Hbf   
3          1   ICE 1163        ICE  Berlin Gesundbrunnen    Stuttgart Hbf   
4          1   ICE 1163        ICE  Berlin Gesundbrunnen    Stuttgart Hbf   
..       ...        ..